# Code Initialization

In [1]:
import stim
import random
import csv
import pandas as pd
import collections
import os

from wrappers.polar_wrapper import (
        polar_code_p2, get_logical_error_on_accepted_states, divide_half_list
)

from wrappers.stim_wrapper import (
    convert_i_to_meas_type, generate_circuit_extraction_syndrome_stim, simulate_stim_polar_code, noisy_cx, noisy_h, noisy_x, noisy_z, 
    noisy_reset, noisy_measurement, calculate_logical_error_result_polar
    
)


# Stim Tableau

In [2]:
print("stim version :", stim.__version__)
sim = stim.TableauSimulator()


stim version : 1.15.0


In [3]:
counts = {}

for i in range(10000):
    # Step-by-step simulation using TableauSimulator
    sim = stim.TableauSimulator()
    p_error = 0.01

    noisy_h(sim, 0, p_error)
    noisy_cx(sim, 0, 1, p_error)
    noisy_cx(sim, 1, 2, p_error)
    noisy_cx(sim, 2, 3, p_error)

    # res0, kick0 = sim.measure_kickback(0)
    # print("Res:", res0, ", Kick:", kick0)
    
    # res1, kick1 = sim.measure_kickback(4)
    # print("Res:", res1, ", Kick:", kick1)

    final_measurements = sim.measure_many(0, 1, 2, 3)
    final_measurements

    bit_string = ''.join(['1' if b else '0' for b in final_measurements])
    bit_string

    if bit_string in counts:
        counts[bit_string] += 1
    else:
        counts[bit_string] = 1

counts

{'1111': 4935,
 '0000': 4827,
 '1000': 51,
 '0010': 22,
 '0100': 18,
 '1011': 13,
 '1101': 7,
 '1100': 33,
 '0001': 14,
 '1110': 15,
 '0111': 41,
 '0011': 24}

## Polar Code

In [4]:
# str_data = "100"
# shots = 1000000

# seed_list = []
# for _ in range(shots):
#     seed_simulator = random.randint(1, 99999999)
#     seed_list.append(seed_simulator)

# df_result = pd.DataFrame({
#         'seed_list': seed_list,
#     })

# str_data = "1m"

# df_result.to_csv(f"./output/STIM/seed_list_{str_data}_3.csv")

In [5]:
def check_for_mismatch(syndrome_bits, n, x_ind, detection_layer = 1):
    """
    Helper function to check for a pattern mismatch in the measurement record.
    Returns True if a mismatch is found, False otherwise.
    d_layer = detection layer
    """
    expected_length = 2**(n - 1)
    
    full_bitstring = ''.join(['1' if b else '0' for b in syndrome_bits])
    
    # Extract the relevant portion of the syndrome for this layer.
    start_index = expected_length * detection_layer
    syndrome_to_check = full_bitstring[start_index:]
    
    bit_gap = 2**x_ind

    # print(len(full_bitstring), full_bitstring, syndrome_to_check, expected_length, bit_gap, x_ind)
    
    for i in range(expected_length):
        if (i % (2 * bit_gap)) < bit_gap:
            j = i + bit_gap
            if syndrome_to_check[i] != syndrome_to_check[j]:
                print(f"Mismatch found at indices ({i}, {j}) for string {syndrome_to_check}.")
                return True
            
    return False

In [6]:
def simulate_stim_polar_code(n, lstate, sim_type, i, p_error, shots, seeds):
    """
    Simulates a quantum circuit for stabilizer code, simplified and optimized.

    Args:
        n (int): code length for polar code
        lstate (str): z for logical |0>, z for logical |+>
        sim_type: normal, m1, m2, m3
        p_error (float): The probability of an error occurring.
        shots (int): The number of simulation runs.
        seeds (list): A list of seeds for the simulator.
        i (int): An index used to determine measurement types (message location)

    Returns:
        dict: A dictionary of measurement outcome counts.
    """
    #m1 circuit simplify
    #m2 m1 + error detection
    #m3 m1 + error correction
    
    meas_type = convert_i_to_meas_type(i, n, lstate)
    N = 2**n
    ancilla_qubits = 2**(n - 1)

    counts = collections.Counter()
    bitstrings = []
    total_qubits = N + ancilla_qubits

    x_ind = 0
    for idx, m_type in enumerate(meas_type):
        if m_type == "x":
            x_ind = idx
            break

    # print("index of the first x :", x_ind)
    # gap = 2**x_ind

    count_detect_discard = 0

    for shot_idx in range(shots):
        sim = stim.TableauSimulator(seed=seeds[shot_idx])

        error_detected_this_shot = False

        for level in range(1, n+1):

            # print(level, "-", meas_type[level - 1])
            
            num_loops = 2**(n - level)
            start_idx_mult = 2**(level) + 2**(level - 1)

            if sim_type in ["m1", "m2"] and level == x_ind + 1:
                x_first = True
            else:
                x_first = False

            if level <= x_ind:
                # skip the beginning of zz measurement
                # print("skip :" , level, x_ind)
                pass
            else:
                # print("num_loops :", num_loops)
                for loop_idx in range(num_loops):
                    start_idx = loop_idx * start_idx_mult
                    generate_circuit_extraction_syndrome_stim(sim, level, meas_type[level - 1], x_first, p_error=p_error, start_idx=start_idx)
            
                # with the simplification, the first measurement will be always 0
                if not x_first:
                    for qb in range(2, total_qubits, 3): 
                        noisy_measurement(sim, qb, p_error)
                        noisy_reset(sim, qb, p_error)

            # add error detection
            # first error detection
            if sim_type in ["m2"] and level - x_ind == 2:
                cms = sim.current_measurement_record()
                # print(cms)

                detection_layer = 1
                if sim_type == "m2":
                    detection_layer -=1

                if check_for_mismatch(cms, n, x_ind, detection_layer):
                    error_detected_this_shot = True
                    break # Break from the `level` loop

        # if error is detected, skip the operation
        if error_detected_this_shot:
            count_detect_discard += 1
            continue # Continue to the next shot

        # Final measurements, simplified.
        for qb_idx in (j for j in range(total_qubits) if j % 3 != 2):
            if lstate == "x":
                noisy_h(sim, qb_idx, p_error)
            noisy_measurement(sim, qb_idx, p_error, )
            

        final_measurements = sim.current_measurement_record()

        # Generate the bit string, reverse it, and update the counts.
        bit_string = ''.join(['1' if b else '0' for b in final_measurements])[::-1]

        if sim_type in ["m1", "m2"]:
            bit_string = bit_string + "0"*(2**(level - 1))
        

        # counts[bit_string] += 1
        bitstrings.append(bit_string)

    # return counts
    return bitstrings

In [7]:
n = 4
lstate = "x"
sim_type = "m2"
p_error = 0
i = 2
shots = 20
# seed_list = range(shots)
seed_list = range(shots)

results = simulate_stim_polar_code(n, lstate, sim_type, i, p_error, shots, seed_list)

counts = collections.Counter(results)
# print(len(results[0]), (results[0]))

zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
zpos_list[n] = i-1
count_accept_n, count_logerror, count_undecided, ler_n, detect_normal, decoding_normal = get_logical_error_on_accepted_states(n, lstate.upper(), counts, zpos_list)

print(count_accept_n, shots - len(results), count_accept_n / len(results), 1 - ler_n, len(results[0]))

# list_8 = []

# for c in counts:
#     list_8.append(c[-16:-8])
#     # list_8.append(c[0:8])

# counts_8 = collections.Counter(list_8)
# counts_8


calculate_logical_error_result_polar(n, lstate, i, p_error, sim_type, shots)

4 x 2 0000 ['x', 'x', 'x', 'x']
20 0 1.0 0.0 48
4 x 2 0000 ['x', 'x', 'x', 'x']


In [9]:
counts

Counter({'110011001111110000110000000000111111001100000000': 1,
         '000000110011001100110000001100000011111100000000': 1,
         '000000111100111111001100001100110011110000000000': 1,
         '001111001111110011000000111100111111001100000000': 1,
         '110011111100000000001111001111001100110000000000': 1,
         '111111110000001111111100000000110000001100000000': 1,
         '110000110011110011111111111111111111111100000000': 1,
         '000011111100000011001111111111000000110000000000': 1,
         '111111001100110000110000001100000011111100000000': 1,
         '110000001111000000110000110011111100000000000000': 1,
         '000000001100110011001100000000000000111100000000': 1,
         '000000001100111111001111000000110000110000000000': 1,
         '000000000011111100111111000011000000110000000000': 1,
         '001111001100110011110000111100001111111100000000': 1,
         '110000000011000011110000110000111100110000000000': 1,
         '110011001100111100000011000000

In [ ]:
# for key_8 in counts_8.keys():
#     print(key_8)

In [ ]:
def simulate_batch_and_save_result_polar(n, lstate, sim_type, p_error, i, shots):

    zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
    zpos_list[n] = i-1

    file_path = f"./output/STIM/polar_n{n}_{lstate}_{i}_{p_error}_{sim_type}.txt"

    existing_data = 0
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            lines = f.readlines()
            existing_data = len(lines)

    # shots = 1000
    seed_list = range(existing_data, existing_data + shots + 1)

    print(existing_data, seed_list[0], seed_list[-1])

    results = simulate_stim_polar_code(n, lstate, sim_type, i, p_error, shots, seed_list)

    with open(file_path, "a") as f:  # "a" means append mode
        f.write("\n".join(results) + "\n")

# count_accept_n, count_logerror, count_undecided, ler_n, detect_normal, decoding_normal = get_logical_error_on_accepted_states(n, lstate.upper(), counts_normal, zpos_list)
# print(sim_type, count_accept_n / shots, 1 - ler_n)



In [ ]:
n = 4
lstate = "x"
sim_type = "normal"
p_error = 0.01
i = 7
shots = int(1e2)

simulate_batch_and_save_result_polar(n, lstate, sim_type, p_error, i, shots)

200 200 100200
7 1010 ['z', 'x', 'z', 'x'] 4
index of the first x : 1


#### Retreive result

In [ ]:
n = 4
lstate = "x"
i = 1
p_error = 0.01
sim_type = "m1"

def calculate_logical_error_result_polar(n, lstate, i, p_error, sim_type):
    file_path = f"./output/STIM/polar_n{n}_{lstate}_{i}_{p_error}_{sim_type}.txt"

    if not os.path.exists(file_path):
        return None  # skip missing files

    with open(file_path, "r") as f:
        lines = f.read().splitlines()

    zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
    zpos_list[n] = i - 1

    counts = collections.Counter(lines)

    count_accept_n, count_logerror, count_undecided, ler_n, detect_normal, decoding_normal = \
        get_logical_error_on_accepted_states(
            n, lstate.upper(), counts, zpos_list
        )

    # Return structured result
    return {
        "n": n,
        "lstate": lstate,
        "i": i,
        "p_error": p_error,
        "sim_type": sim_type,
        "count_accept": count_accept_n,
        "count_logerror": count_logerror,
        "count_undecided": count_undecided,
        "LER": ler_n,
        "detect_normal": detect_normal,
        "decoding_normal": decoding_normal,
    }



1000000
